# GAN Evaluation — FID, Inception Score, Precision/Recall

Up to Module 20 we built generators. Now the practical engineering question: **how do we know whether a trained GAN is actually good?**

GAN training losses are famously uninformative. A falling Generator loss doesn't guarantee better images, and a flat Discriminator loss doesn't guarantee stability. The community developed *distribution-level* metrics instead.

Three properties matter:

- **Quality** — are generated images realistic?
- **Diversity** — do generated images differ from each other?
- **Coverage** — does the Generator represent all the modes of the real distribution?

This notebook trains a quick DCGAN on MNIST, then evaluates it with:

1. **FID** — Fréchet Inception Distance between real and generated feature distributions
2. **Inception Score** — classifier confidence × class diversity on generated samples
3. **Precision / Recall** — quality vs coverage diagnostic, useful for spotting mode collapse
4. **Diversity** — average pairwise feature distance among generated samples

We use a small CNN trained on MNIST as the feature extractor / classifier. The standard FID recipe uses Inception v3 pretrained on ImageNet, but for MNIST that's a poor fit. A domain-matched feature extractor is the more honest choice here.

## Implementation Plan

- **DCGAN generator + discriminator** on MNIST (small DCGAN, 10 epochs to keep runtime reasonable).
- **MNIST feature extractor / classifier**: a small CNN trained for 2 epochs to get a useful feature space. We then use the penultimate activations as our embedding for FID and precision/recall, and the final softmax for Inception Score.
- **FID**: extract features for both real and generated images, fit multivariate Gaussians (mean + covariance), compute `||mu_r - mu_g||^2 + Tr(sigma_r + sigma_g - 2 sqrt(sigma_r sigma_g))`.
- **Inception Score**: `exp(mean over samples of KL(p(y|x) || p(y)))`. Higher is better.
- **Precision / Recall (improved version, Kynkäänniemi et al.)**: for each query point, the distance to the k-th nearest neighbor defines whether it lies in the other manifold. Precision = fraction of generated points inside the real manifold; Recall = fraction of real points inside the generated manifold.
- **Diversity**: average pairwise feature distance among generated samples.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
import numpy as np
import os

torch.manual_seed(42)
np.random.seed(42)

## 1. Setup and Hyperparameters

Two training runs: the GAN itself (10 epochs) and the small classifier that supplies the feature extractor (2 epochs).

In [ ]:
latent_dim = 100
img_channels = 1
img_size   = 64
features_g = 64
features_d = 64
batch_size = 64
lr         = 2e-4
betas      = (0.5, 0.999)
gan_epochs       = 10
classifier_epochs = 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

os.makedirs('samples_Eval', exist_ok=True)

## 2. Data — MNIST at 64x64

In [ ]:
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

dataloader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=transform),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

## 3. Weight Initialization

In [ ]:
def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

## 4. DCGAN Generator and Discriminator

Standard DCGAN architecture — the same as `02_DCGAN.ipynb`, kept short here so the focus is on evaluation.

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_channels=1, features_g=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, features_g * 8, 4, 1, 0, bias=False),
            nn.BatchNorm2d(features_g * 8), nn.ReLU(True),

            nn.ConvTranspose2d(features_g * 8, features_g * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 4), nn.ReLU(True),

            nn.ConvTranspose2d(features_g * 4, features_g * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g * 2), nn.ReLU(True),

            nn.ConvTranspose2d(features_g * 2, features_g, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_g), nn.ReLU(True),

            nn.ConvTranspose2d(features_g, img_channels, 4, 2, 1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        return self.net(z.view(z.size(0), z.size(1), 1, 1))


class Discriminator(nn.Module):
    def __init__(self, img_channels=1, features_d=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(img_channels, features_d, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(features_d, features_d * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 2), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(features_d * 2, features_d * 4, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 4), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(features_d * 4, features_d * 8, 4, 2, 1, bias=False),
            nn.BatchNorm2d(features_d * 8), nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(features_d * 8, 1, 4, 1, 0, bias=False),
        )

    def forward(self, x):
        return self.net(x).view(x.size(0), -1)

## 5. Train the GAN

Standard alternating training loop with `BCEWithLogitsLoss`. We deliberately use a small number of epochs (10) — enough for a partially-trained generator that produces recognizable digits, fast enough to keep the notebook runnable end-to-end. Evaluation metrics on a 10-epoch GAN are still meaningful: they show the *behavior* of each metric, even if the absolute numbers are not state-of-the-art.

In [ ]:
G = Generator(latent_dim=latent_dim, img_channels=img_channels, features_g=features_g).to(device)
D = Discriminator(img_channels=img_channels, features_d=features_d).to(device)
G.apply(weights_init); D.apply(weights_init)

optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=betas)
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=betas)
criterion = nn.BCEWithLogitsLoss()

fixed_noise = torch.randn(64, latent_dim, device=device)

G.train(); D.train()
for epoch in range(gan_epochs):
    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.to(device)
        b = real_imgs.size(0)
        valid = torch.ones(b, 1, device=device)
        fake_t = torch.zeros(b, 1, device=device)

        # D
        optimizer_D.zero_grad()
        real_logits = D(real_imgs)
        d_real_loss = criterion(real_logits, valid)
        z = torch.randn(b, latent_dim, device=device)
        with torch.no_grad():
            gen_imgs = G(z)
        fake_logits = D(gen_imgs)
        d_fake_loss = criterion(fake_logits, fake_t)
        d_loss = (d_real_loss + d_fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        # G
        optimizer_G.zero_grad()
        gen_imgs = G(z)
        validity_logits = D(gen_imgs)
        g_loss = criterion(validity_logits, valid)
        g_loss.backward()
        optimizer_G.step()

    G.eval()
    with torch.no_grad():
        sample = G(fixed_noise).detach().cpu()
    save_image(sample, f"samples_Eval/epoch_{epoch+1:02d}.png", nrow=8, normalize=True)
    G.train()
    print(f"Epoch [{epoch+1}/{gan_epochs}] done")

G.eval()
print('GAN training done.')

## 6. Random Samples — Visual Inspection

Before any metric: just look at the samples. Visual inspection is the *first* evaluation, even though it's not scalable.

In [ ]:
with torch.no_grad():
    samples = G(fixed_noise).cpu()

grid = make_grid(samples, nrow=8, normalize=True)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
plt.axis('off')
plt.title('Trained GAN — random samples')
plt.show()

## 7. Feature Extractor / Classifier

We need a network that maps images to a useful embedding — and a classifier head for Inception Score. We train a small CNN on MNIST for 2 epochs (just enough to give us a reasonable embedding). Its penultimate layer is the feature vector; its final softmax gives `p(y | x)` for IS.

In a real experiment you'd use a *much* larger classifier, possibly pretrained on ImageNet. For MNIST, this small CNN is a more honest match.

In [ ]:
class FeatureExtractor(nn.Module):
def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(32, 64, 3, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),                          # 32x32

            nn.Conv2d(64, 128, 3, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.MaxPool2d(2),                          # 16x16

            nn.Conv2d(128, 128, 3, padding=1), nn.LeakyReLU(0.2, inplace=True),
            nn.AdaptiveAvgPool2d(4),                  # 4x4
        )
        self.feature_dim = 128 * 4 * 4
        self.fc_feat = nn.Linear(self.feature_dim, 128)
        self.classifier = nn.Linear(128, 10)

    def features(self, x):
        h = self.conv(x).view(x.size(0), -1)
        return F.leaky_relu(self.fc_feat(h), 0.2, inplace=True)

    def logits(self, x):
        return self.classifier(self.features(x))

    def forward(self, x):
        return self.logits(x)


feat_net = FeatureExtractor().to(device)
optimizer_F = optim.Adam(feat_net.parameters(), lr=1e-3)
ce_loss = nn.CrossEntropyLoss()

feat_net.train()
for epoch in range(classifier_epochs):
    for imgs, labels in dataloader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer_F.zero_grad()
        logits = feat_net(imgs)
        loss = ce_loss(logits, labels)
        loss.backward()
        optimizer_F.step()
    print(f"Classifier epoch [{epoch+1}/{classifier_epochs}]  CE: {loss.item():.3f}")

feat_net.eval()
print('Feature extractor trained.')

## 8. Build Feature Sets

We need two feature sets: one for **real** MNIST images and one for **generated** samples. We use the same number of samples for both so the FID is directly comparable.

In [ ]:
n_eval = 2000

# Real features
real_feats = []
real_labels = []
with torch.no_grad():
    for imgs, labels in dataloader:
        real_feats.append(feat_net.features(imgs.to(device)))
        real_labels.append(labels.to(device))
        if sum(f.size(0) for f in real_feats) >= n_eval:
            break
real_feats  = torch.cat(real_feats, dim=0)[:n_eval]
real_labels = torch.cat(real_labels, dim=0)[:n_eval]

# Generated features
with torch.no_grad():
    z_eval = torch.randn(n_eval, latent_dim, device=device)
    gen_imgs_eval = G(z_eval)
    fake_feats = feat_net.features(gen_imgs_eval)
    fake_logits = feat_net(gen_imgs_eval)
    fake_labels = fake_logits.argmax(dim=1)

print(f'real_feats:  {tuple(real_feats.shape)}')
print(f'fake_feats:  {tuple(fake_feats.shape)}')

## 9. FID — Fréchet Inception Distance

Two steps:

1. Fit multivariate Gaussians to the real and generated feature sets (mean + covariance).
2. Compute the Fréchet distance between those two Gaussians.

Formula:

```
FID = ||mu_r - mu_g||^2 + Tr(sigma_r + sigma_g - 2 * sqrt(sigma_r @ sigma_g))
```

Lower FID = closer feature distributions.

Note: the covariance estimate can be unstable for small sample counts. We add a small diagonal `eps * I` for numerical safety.

In [ ]:
def matrix_sqrt(M):
    """Symmetric matrix square root via eigendecomposition."""
    M = (M + M.T) / 2
    eigvals, eigvecs = torch.linalg.eigh(M)
    eigvals = torch.clamp(eigvals, min=0)
    return eigvecs @ torch.diag(eigvals.sqrt()) @ eigvecs.T


def compute_fid(real_feats, fake_feats, eps=1e-3):
    mu_r = real_feats.mean(dim=0)
    mu_g = fake_feats.mean(dim=0)

    diff = (mu_r - mu_g).pow(2).sum().item()

    sigma_r = torch.cov(real_feats.T) + eps * torch.eye(real_feats.size(1), device=real_feats.device)
    sigma_g = torch.cov(fake_feats.T) + eps * torch.eye(fake_feats.size(1), device=fake_feats.device)

    covmean = matrix_sqrt(sigma_r @ sigma_g)
    if torch.is_complex(covmean):
        covmean = covmean.real

    trace_term = torch.trace(sigma_r) + torch.trace(sigma_g) - 2 * torch.trace(covmean)
    fid = diff + trace_term.item()
    return fid


fid_value = compute_fid(real_feats, fake_feats)
print(f'FID: {fid_value:.3f}')

## 10. Inception Score (IS)

The Inception Score uses the classifier's softmax outputs:

```
IS = exp(mean over samples of KL(p(y|x) || p(y)))
```
where `p(y|x)` is the classifier's predicted distribution for sample `x` and `p(y)` is the marginal class distribution across all generated samples.

Higher IS = samples are individually classifiable AND collectively diverse.

In [ ]:
def compute_inception_score(fake_logits, n_split=10, eps=1e-8):
    probs = F.softmax(fake_logits, dim=1).cpu().numpy()
    n = probs.shape[0]
    scores = []
    for i in range(n_split):
        part = probs[i * (n // n_split): (i + 1) * (n // n_split), :]
        py = np.mean(part, axis=0)
        kl = part * (np.log(part + eps) - np.log(py + eps))
        kl = np.mean(np.sum(kl, axis=1))
        scores.append(np.exp(kl))
    return float(np.mean(scores)), float(np.std(scores))


is_mean, is_std = compute_inception_score(fake_logits)
print(f'Inception Score: {is_mean:.3f} ± {is_std:.3f}')

## 11. Precision and Recall (Kynkäänniemiemi-style)

Improved precision/recall defines the manifold of a set by the distance to the k-th nearest neighbor. A point lies inside the manifold if its distance to the k-th nearest neighbor is below a threshold (typically the k-th nearest neighbor distance of the *other* set's points).

**Precision** = fraction of fake points inside the real manifold.
**Recall** = fraction of real points inside the fake manifold.

A GAN with high quality and low diversity will have **high precision, low recall** (the textbook mode-collapse signature).

In [ ]:
def pairwise_distances(X):
    # X: (N, D). Returns (N, N) Euclidean distances.
    X = X.float()
    sq = (X * X).sum(dim=1, keepdim=True)              # (N, 1)
    dist = sq + sq.T - 2 * X @ X.T
    return torch.clamp(dist, min=0).sqrt()


def precision_recall(real_feats, fake_feats, k=3):
    """Kynkäänniemiemi et al. precision/recall on the k-NN manifold."""
    real_feats = real_feats.float()
    fake_feats = fake_feats.float()

    # Distance to k-th nearest neighbor for each point in its OWN set
    dist_real = pairwise_distances(real_feats)
    dist_fake = pairwise_distances(fake_feats)
    dist_real.fill_diagonal_(float('inf'))
    dist_fake.fill_diagonal_(float('inf'))
    radii_real = dist_real.kthvalue(k, dim=1).values    # (N_r,)
    radii_fake = dist_fake.kthvalue(k, dim=1).values    # (N_f,)

    # For each point in the QUERY set, find distance to nearest point in the REFERENCE set
    dist_r2f = pairwise_distances(real_feats @ torch.eye(real_feats.size(1), device=real_feats.device).T  # dummy
                                   + fake_feats * 0)  # avoid issues
    # Cleaner: compute cross distances directly
    cross_rf = torch.cdist(real_feats, fake_feats)     # (N_r, N_f)
    cross_fr = cross_rf.T                              # (N_f, N_r)

    # For each real point, is there a fake point within the real radii? -> inside fake manifold
    nn_fake_for_real = cross_rf.min(dim=1).values       # (N_r,)
    inside_fake_manifold = (nn_fake_for_real <= radii_fake.unsqueeze(0)).any(dim=1).float()
    recall = inside_fake_manifold.mean().item()

    # For each fake point, is there a real point within the fake radii? -> inside real manifold
    nn_real_for_fake = cross_fr.min(dim=1).values       # (N_f,)
    inside_real_manifold = (nn_real_for_fake <= radii_real.unsqueeze(0)).any(dim=1).float()
    precision = inside_real_manifold.mean().item()

    return precision, recall


precision, recall = precision_recall(real_feats, fake_feats, k=3)
print(f'Precision: {precision:.3f}')
print(f'Recall:    {recall:.3f}')

## 12. Diversity — Pairwise Feature Distance Among Generated Samples

The simplest diversity diagnostic: average pairwise L2 distance among the generated samples in feature space. Low diversity + high precision = mode-collapse warning sign.

In [ ]:
with torch.no_grad():
    dist_fake = pairwise_distances(fake_feats)
    dist_fake.fill_diagonal_(float('inf'))
    diversity = dist_fake.mean().item()

# Compare with real diversity (sanity check that the metric is meaningful)
    dist_real = pairwise_distances(real_feats)
    dist_real.fill_diagonal_(float('inf'))
    real_diversity = dist_real.mean().item()

print(f'Generated diversity: {diversity:.3f}')
print(f'Real diversity:      {real_diversity:.3f}')
print(f'Diversity ratio (gen / real): {diversity / real_diversity:.3f}')

## 13. Class Distribution Comparison

Real MNIST is roughly uniform across the 10 digit classes. The Generator's predicted class distribution should approach that — if it doesn't, you've got mode imbalance.

In [ ]:
real_class_dist = torch.bincount(real_labels, minlength=10).float()
real_class_dist = real_class_dist / real_class_dist.sum()

fake_class_dist = torch.bincount(fake_labels, minlength=10).float()
fake_class_dist = fake_class_dist / fake_class_dist.sum()

x = np.arange(10)
width = 0.4
plt.figure(figsize=(8, 4))
plt.bar(x - width / 2, real_class_dist.cpu().numpy(), width, label='Real MNIST', color='tab:blue', alpha=0.7)
plt.bar(x + width / 2, fake_class_dist.cpu().numpy(), width, label='Generated', color='tab:orange', alpha=0.7)
plt.xticks(x)
plt.xlabel('Predicted class')
plt.ylabel('Fraction')
plt.title('Predicted class distribution')
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.show()

## 14. Summary Table — All Metrics in One Place

In [ ]:
print(f'{"Metric":<35} {"Value":>12}')
print('-' * 48)
print(f'{"FID (lower is better)":<35} {fid_value:>12.3f}')
print(f'{"Inception Score (higher is better)":<35} {is_mean:>12.3f}')
print(f'{"Precision (higher is better)":<35} {precision:>12.3f}')
print(f'{"Recall (higher is better)":<35} {recall:>12.3f}')
print(f'{"Diversity (gen)":<35} {diversity:>12.3f}')
print(f'{"Diversity (real)":<35} {real_diversity:>12.3f}')
print(f'{"Diversity ratio (gen / real)":<35} {diversity / real_diversity:>12.3f}')

## Recap — How to Think About Each Metric

| Metric | Measures | Better when | Pitfall |
|---|---|---|---|
| **FID** | feature-distribution distance | lower | sensitive to sample size, feature extractor, preprocessing |
| **Inception Score** | per-sample classifier confidence + class diversity | higher | doesn't compare to real data; can reward unrealistic but classifiable samples |
| **Precision** | fraction of generated samples inside the real manifold | higher | mode collapse -> high precision, low recall |
| **Recall** | fraction of real samples inside the generated manifold | higher | low recall = poor coverage of real distribution |
| **Diversity** | average pairwise feature distance among generated samples | higher | low diversity = collapsed generator |

**What changed and why.**

- **Single-number metrics compress too much.** A GAN with high FID can still be excellent at one mode and terrible at others. Precision/recall separates the two failure modes.
- **FID assumes Gaussians.** That assumption is convenient but wrong. The actual feature distributions are not Gaussian, especially for high-quality generators that produce tight clusters. FID is a *proxy*, not a ground-truth measure.
- **Inception Score does not use real data.** It only sees the generated samples. A generator that produces *recognizable* but *unrealistic* samples can score well. FID fixes this by comparing to a real-data reference.
- **Sample size matters.** All these metrics are *finite-sample estimates* of true distribution properties. Reporting them requires reporting the sample count.
- **Domain-matched features matter.** Using ImageNet-pretrained Inception for MNIST evaluation is suboptimal. We trained a small MNIST classifier instead — the same architecture pattern, a more honest feature space for this dataset.
- **Visual inspection still matters.** None of these metrics is a substitute for looking at samples. The summary table gives you numbers; the sample grid gives you understanding.

**Common implementation pitfalls** — quick reference:

- **Matrix square root can be numerically unstable.** Use `torch.linalg.eigh` on a symmetrized matrix and clamp negative eigenvalues to zero. This is what `matrix_sqrt` does here.
- **`torch.cov` requires a feature-dim matrix and produces a covariance matrix of shape (D, D).** Make sure you transpose `real_feats` correctly — `torch.cov(real_feats.T)`.
- **Add a small `eps * I` to covariance** for numerical safety when sample count is small or feature dim is large. Without it, FID can be `NaN`.
- **`fake_labels.argmax(dim=1)` is what the classifier thinks the digit is.** If your classifier isn't well-trained, this is meaningless. Train the classifier adequately before using its outputs.
- **Precision/Recall with too small `k` is noisy.** `k = 3` is a reasonable starting point. Larger `k` smooths the manifold estimate.
- **Diversity is meaningful only relative to real diversity.** An absolute number doesn't tell you if the Generator is collapsed — only the *ratio* between generated and real diversity does.

**The takeaway.** No single metric captures whether a GAN is good. FID is the standard, but it is a *proxy* and it depends on the evaluation protocol. Precision/recall separates quality from coverage. Diversity catches collapse. Visual inspection catches things no metric can. Use them together.